# Preprocessament

Aquest notebook construeix una taula de **característiques per frame** per preparar l'algoritme de possessió. Treballa amb el **tracking ATD tàctic** i l'**Smart Tagging automàtic**.

**No entrena cap model, no utilitza el ground truth i no fa etiquetatge manual.** Tampoc no obre el vídeo ni envia les dades a Internet.

**Resultat:** `data/processed/features_tactical.csv` (o a la carpeta externa configurada amb `MTAH_DATA_DIR`), amb una fila per cadascun dels **199.319 frames** i una sèrie de variables explicatives. **No és encara el CSV de submissió**: no hi ha cap columna `label`.


## 1. Configuració i importacions

In [ ]:
from pathlib import Path
import sys
import re
import unicodedata
import numpy as np
import pandas as pd

# Si tens les dades en una ubicació externa, posa aquí la carpeta que conté tactical/.
DATA_DIR_OVERRIDE = None  # Exemple: r"D:\HackathonDades"
WRITE_CSV = True

# Detectem l'arrel del repositori independentment de si executeu des de l'arrel o solution/.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "games.py").is_file() and (p / "starter").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError(
        "No trobo games.py ni starter/. Obre el notebook des de l'arrel del repositori o solution/."
    )
sys.path.insert(0, str(ROOT))

from games import GAME, GAMES, DATA_DIR, source_frames
from starter import baseline, load_atd, load_smart_tagging

FPS = GAMES[GAME]["fps"]
PITCH_LENGTH_M, PITCH_WIDTH_M = GAMES[GAME]["pitch"]
N_FRAMES = source_frames("tactical", GAME)
DATA_ROOT = Path(DATA_DIR_OVERRIDE or DATA_DIR).expanduser().resolve()
OUTPUT_PATH = DATA_ROOT / "processed" / "features_tactical.csv"

print(f"Joc: {GAME} | FPS: {FPS} | Frames esperats: {N_FRAMES:,}")
print("Dades en ubicació local configurada. No es farà cap petició de xarxa.")

## 2. Lectura eficient del tracking

Un registre ATD té el format `frame_id:jugador;jugador;...:pilota`. Els punts observats tenen coordenades normalitzades (`x`, `y`) i velocitat (`s`); els punts perduts contenen `NaN`.

Carregarem el TXT **en streaming**, com fa el baseline oficial, perquè el loader complet de Kloppy pot consumir molta RAM. Utilitzarem les metadades XML únicament per associar cada *track slot* a l'equip `H` o `A`.

**Important:** `Track 1..46` són fragments de seguiment (*tracklets*), no identitats personals estables. Les distàncies es calcularan en metres, multiplicant `x` i `y` per les dimensions del camp. Quan no hi ha observació, conservem `NaN`; **no inferim possessió ni fora de joc**.

In [ ]:
def parse_xyz(token):
    """Converteix 'x,y,s' (o 'NaN') a tres floats; s és opcional."""
    parts = token.strip().split(",")
    if len(parts) < 2:
        return np.nan, np.nan, np.nan
    try:
        x = float(parts[0])
        y = float(parts[1])
        speed = float(parts[2]) if len(parts) >= 3 else np.nan
        return x, y, speed
    except ValueError:
        raise ValueError("Registre ATD inesperat: coordenades no numèriques")


def parse_tracking_lines(lines, team_by_slot, n_frames, pitch, fps):
    """Crea features a partir de línies ATD; no usa ground truth ni models."""
    length, width = pitch
    rows = []
    expected_frame = 1
    team_by_slot = np.asarray(team_by_slot)
    if not np.isin(team_by_slot, [0, 1]).all():
        raise ValueError("team_by_slot ha de contenir 0=H o 1=A")

    for line in lines:
        if not line.strip():
            continue
        parts = line.rstrip("\r\n").split(":")
        if len(parts) != 3:
            raise ValueError(f"Format ATD incorrecte a la línia {expected_frame}")
        frame_id = int(parts[0])
        if frame_id != expected_frame:
            raise ValueError(f"Frames desalineats: esperava {expected_frame} i he rebut {frame_id}")
        frame = frame_id - 1  # ATD 1-based -> vídeo/submissió 0-based
        expected_frame += 1

        players = parts[1].split(";")
        if len(players) != len(team_by_slot):
            raise ValueError(
                f"Frame {frame}: {len(players)} slots al TXT, "
                f"{len(team_by_slot)} a les metadades XML"
            )

        bx, by, bs = parse_xyz(parts[2])
        visible = np.isfinite(bx) and np.isfinite(by)
        bx_m = bx * length if visible else np.nan
        by_m = by * width if visible else np.nan
        home_min = away_min = np.inf
        home_speed = away_speed = np.nan
        n_home = n_away = n_home_2m = n_away_2m = 0

        for slot, raw_player in enumerate(players):
            px, py, ps = parse_xyz(raw_player)
            if not (np.isfinite(px) and np.isfinite(py)):
                continue
            team = int(team_by_slot[slot])
            if team == 0:
                n_home += 1
            else:
                n_away += 1
            if not visible:
                continue
            distance = np.hypot((px - bx) * length, (py - by) * width)
            if team == 0:
                n_home_2m += distance <= 2.0
                if distance < home_min:
                    home_min, home_speed = distance, ps
            else:
                n_away_2m += distance <= 2.0
                if distance < away_min:
                    away_min, away_speed = distance, ps

        rows.append((frame, bx, by, bs, visible,
                     bx_m, by_m, n_home, n_away,
                     home_min if np.isfinite(home_min) else np.nan,
                     away_min if np.isfinite(away_min) else np.nan,
                     home_speed, away_speed, n_home_2m, n_away_2m))

    if len(rows) != n_frames:
        raise ValueError(f"Fitxer incomplet: he llegit {len(rows)} frames i n'esperava {n_frames}")

    cols = ["frame", "ball_x_norm", "ball_y_norm", "ball_speed_mps_atd",
            "ball_visible", "ball_x_m", "ball_y_m", "n_home_tracked", "n_away_tracked",
            "home_nearest_m", "away_nearest_m", "home_nearest_speed_mps",
            "away_nearest_speed_mps", "n_home_within_2m", "n_away_within_2m"]
    df = pd.DataFrame.from_records(rows, columns=cols)
    df["frame"] = df["frame"].astype("int32")
    df["ball_visible"] = df["ball_visible"].astype(bool)
    df["time_s"] = df["frame"] / fps

    # Marge positiu = jugador local més pròxim que el visitant (si tots dos existeixen).
    df["distance_margin_away_minus_home_m"] = df["away_nearest_m"] - df["home_nearest_m"]
    home = df["home_nearest_m"].to_numpy(dtype=float)
    away = df["away_nearest_m"].to_numpy(dtype=float)
    nearest = np.full(len(df), "", dtype="<U1")
    nearest[np.isfinite(home) & ((home <= away) | ~np.isfinite(away))] = "H"
    nearest[np.isfinite(away) & ((away < home) | ~np.isfinite(home))] = "A"
    df["nearest_team_observed"] = nearest  # '' si no es pot determinar

    # No inventem posicions: el desplaçament només s'estima entre frames consecutius visibles.
    bx_arr = df["ball_x_m"].to_numpy(dtype=float)
    by_arr = df["ball_y_m"].to_numpy(dtype=float)
    displacement = np.full(len(df), np.nan, dtype=float)
    good_pairs = np.isfinite(bx_arr[1:]) & np.isfinite(bx_arr[:-1]) & \
                 np.isfinite(by_arr[1:]) & np.isfinite(by_arr[:-1])
    displacement[1:][good_pairs] = np.hypot(
        bx_arr[1:][good_pairs] - bx_arr[:-1][good_pairs],
        by_arr[1:][good_pairs] - by_arr[:-1][good_pairs]
    )
    df["ball_displacement_m_between_frames"] = displacement
    df["ball_apparent_speed_mps"] = displacement * fps  # no és velocitat 3D real

    # Durada consecutiva de l'absència d'observació, NO durada fora de joc.
    is_missing = ~df["ball_visible"]
    df["ball_missing_streak_frames"] = (
        is_missing.astype(int).groupby((~is_missing).cumsum()).cumsum().astype("int32")
    )
    return df

## 3. Prova amb dades **sintètiques**

Comprovem la conversió de frames, les distàncies, els `NaN` i l'equip més proper **sense tocar dades restringides**.

In [ ]:
from io import StringIO

sample = StringIO("""1:0.20,0.50,2;0.80,0.50,3:0.21,0.50,5
2:0.20,0.50,2;0.80,0.50,3:NaN
3:0.20,0.50,2;0.80,0.50,3:0.78,0.50,6
""")
test_df = parse_tracking_lines(sample, team_by_slot=[0, 1],
                               n_frames=3, pitch=(100.0, 50.0), fps=25)
assert test_df["frame"].tolist() == [0, 1, 2]
assert test_df["ball_visible"].tolist() == [True, False, True]
assert test_df["nearest_team_observed"].tolist() == ["H", "", "A"]
assert np.isclose(test_df.loc[0, "home_nearest_m"], 1.0)
assert test_df["ball_missing_streak_frames"].tolist() == [0, 1, 0]
assert np.isnan(test_df.loc[2, "ball_displacement_m_between_frames"])
print("Proves sintètiques del tracking superades.")

## 4. Llegir el tracking real (en local)

Es fa servir `starter/load_atd.py` per localitzar els fitxers i `starter/baseline.py` per obtenir l'equip de cada *track slot*. No cal `dataset.to_df()` perquè així evitem carregar els ~9 milions de registres de jugador a memòria alhora.

Si surt un error de fitxer no trobat, revisa `DATA_DIR_OVERRIDE` o l'estructura `data/tactical/`.

In [ ]:
# Obtenim les rutes dels dos fitxers que necessitem:
# - meta_path: fitxer XML amb la informació dels equips i jugadors.
# - raw_path: fitxer TXT amb les posicions dels jugadors i la pilota.
meta_path, raw_path = load_atd.atd_paths(
    game=GAME, data_dir=str(DATA_ROOT), source="tactical"
)

# Llegim l'XML per saber a quin equip pertany cada track (identificador de seguiment).
team_by_slot = baseline.team_of_each_track(meta_path)

# Mostrem quants tracks hem trobat.
print(f"S'han detectat {len(team_by_slot)} track slots en les metadades.")


# Obrim el fitxer TXT que conté les dades de tracking.
with open(raw_path, encoding="utf-8-sig") as handle:

    # Llegim les dades i les transformem en un DataFrame.
    # També calculem característiques útils, com distàncies i velocitats.
    features_tracking = parse_tracking_lines(
        handle,
        team_by_slot=team_by_slot,           # Equip al qual pertany cada track
        n_frames=N_FRAMES,                   # Nombre total de frames del vídeo
        pitch=(PITCH_LENGTH_M, PITCH_WIDTH_M), # Dimensions del camp en metres
        fps=FPS                              # Nombre de frames per segon
    )


# Comprovem que hem processat tots els frames del vídeo.
assert len(features_tracking) == N_FRAMES

# Comprovem que el primer frame és el 0.
assert features_tracking["frame"].iloc[0] == 0

# Comprovem que l'últim frame té el número esperat.
assert features_tracking["frame"].iloc[-1] == N_FRAMES - 1

# Si totes les comprovacions són correctes, mostrem aquest missatge.
print("Tracking processat; frames consecutius i sense desalineació.")


In [ ]:
features_tracking.columns

In [ ]:
features_tracking[features_tracking["ball_visible"] == True]

## 5. Afegir Smart Tagging (intervals que poden coincidir)

El carregador oficial retorna `code`, `start_frame`, `end_frame` (final **exclusiu**) i `tags` com un diccionari. L'XML està sincronitzat amb el vídeo tàctic.

Per cada tipus d'etiqueta generem columnes `tag_<nom>_H`, `tag_<nom>_A` i `tag_<nom>_unknown` (valors 0/1). Així **no perdem informació quan dues fases es solapen**. També es creen senyals específics de reinici (`SET PIECES`).

⚠️ **El `Team` d'una etiqueta defensiva no és necessàriament el posseïdor de la pilota.** No convertim automàticament cap etiqueta tàctica en `H`, `A` o `D` de la submissió. I una etiqueta `SET PIECES` és un interval de reinici, **no prova que tot l'interval sigui fora de joc**.

In [ ]:
def slug(text):
    text = unicodedata.normalize("NFKD", str(text)).encode("ascii", "ignore").decode()
    return re.sub(r"[^a-z0-9]+", "_", text.lower()).strip("_")

# Valor Team del Smart Tagging segons DATA.md.
TAG_TEAM_MAP = {
    "barcelona": "H",
    "fc barcelona": "H",
    "atletic club": "A",  # Valor publicat per Metrica
    "athletic club": "A",
    "n/a": "unknown",
    "": "unknown"
}


def attach_smart_tagging(df, events):
    n = len(df)
    codes = sorted({slug(e.get("code") or "unknown") for e in events})
    kinds = ("H", "A", "unknown")
    columns = {
        f"tag_{code}_{kind}": np.zeros(n, dtype=np.uint8)
        for code in codes for kind in kinds
    }
    # Signals auxiliars sense suposar que el joc està aturat a tot el segment.
    restart_types = sorted({slug(e.get("tags", {}).get("Type"))
                            for e in events if slug(e.get("code")) == "set_pieces"
                            and e.get("tags", {}).get("Type")})
    for kind in restart_types:
        columns[f"restart_type_{kind}"] = np.zeros(n, dtype=np.uint8)
    event_count = np.zeros(n, dtype=np.int16)
    invalid = []

    for ev in events:
        begin = max(0, int(ev["start_frame"]))
        end = min(n, int(ev["end_frame"]))  # exclòs
        if begin >= end:
            continue
        code = slug(ev.get("code") or "unknown")
        raw_team = (ev.get("tags", {}).get("Team") or "").strip().casefold()
        if raw_team not in TAG_TEAM_MAP:
            invalid.append(raw_team)
            continue
        team = TAG_TEAM_MAP[raw_team]
        columns[f"tag_{code}_{team}"][begin:end] = 1
        event_count[begin:end] += 1

        if code == "set_pieces":
            kind = slug(ev.get("tags", {}).get("Type") or "")
            if kind and f"restart_type_{kind}" in columns:
                columns[f"restart_type_{kind}"][begin:end] = 1

    if invalid:
        raise ValueError(f"Team de Smart Tagging desconegut: {sorted(set(invalid))}. "
                         "Revisa TAG_TEAM_MAP abans de continuar.")
    columns["tag_event_count"] = event_count
    return pd.concat([df, pd.DataFrame(columns, index=df.index)], axis=1)


# Test sintètic: dues etiquetes solapades, una ofensiva i una defensiva.
test_events = [
    {"code": "BUILD UP", "start_frame": 0, "end_frame": 2, "tags": {"Team": "Barcelona"}},
    {"code": "DEFENSIVE TRANSITION", "start_frame": 1, "end_frame": 3,
     "tags": {"Team": "Atletic Club"}}
]
smart_test = attach_smart_tagging(test_df, test_events)
assert smart_test.loc[1, "tag_build_up_H"] == 1
assert smart_test.loc[1, "tag_defensive_transition_A"] == 1
assert smart_test.loc[1, "tag_event_count"] == 2
print("Proves sintètiques de Smart Tagging superades.")

In [ ]:
events = load_smart_tagging.load(game=GAME, data_dir=str(DATA_ROOT), fps=FPS)
features = attach_smart_tagging(features_tracking, events)
assert features["frame"].is_unique
assert features["frame"].tolist() == list(range(N_FRAMES))
print(f"Smart Tagging integrat: {len(events)} intervals, {len(features.columns)} columnes.")

## 6. Control de qualitat (sense ground truth)

Aquestes comprovacions són automàtiques i no fan servir etiquetes de possessió ni el vídeo. En particular, un `NaN` de pilota és informació absent, **no una etiqueta `D`**.

No mostrem ni exportem exemples de files reals al notebook per reduir el risc d'incloure dades restringides a les sortides del `.ipynb`.

In [ ]:
assert len(features) == N_FRAMES
assert np.array_equal(features["frame"].to_numpy(), np.arange(N_FRAMES))
assert features["ball_visible"].dtype == bool
assert (features["ball_missing_streak_frames"] >= 0).all()
assert features["nearest_team_observed"].isin(["", "H", "A"]).all()
assert (features.loc[~features["ball_visible"], "nearest_team_observed"] == "").all()
assert (features["tag_event_count"] >= 0).all()

# Resum estructural, no dades del partit ni valors de posicions.
print("Controls completats correctament.")
print("Filas:", len(features), "| Columnes:", len(features.columns))
print("Features:", ", ".join(features.columns))

## 7. Exportar a CSV privat

Escrivim el resultat **dins de la carpeta local de dades** (`data/processed/` o la vostra carpeta externa), no a `solution/` ni `evaluation/results/`. L'`data/` del repositori oficial ja està exclòs de Git.

Si el fitxer real està fora del repositori, assegura't que també sigui una ubicació privada. **No incloguis aquest CSV a GitHub ni l'enviïs a un agent IA allotjat al núvol.**

In [ ]:
if WRITE_CSV:
    OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
    features.to_csv(OUTPUT_PATH, index=False, na_rep="")
    print("Features guardades en el directori local de dades configurat:")
    print(OUTPUT_PATH.name)
    print("Aquest fitxer no és una submissió; la fase següent estimarà possessions.")
else:
    print("WRITE_CSV=False: les features només es mantenen a la variable 'features'.")

## 8. RESUM

Aquest dataset inclou per frame: observació de pilota, coordenades i velocitat reportada, distància al jugador observat més proper de cada equip, proximitat, velocitats dels jugadors més propers, nombre de tracks visibles, longitud de les absències de pilota i etiquetes tàctiques que es poden solapar.

La homografia no és necessària de moment perquè el tracking ja proporciona coordenades normalitzades del camp; seria útil quan afegim deteccions extretes del vídeo.